# 01 — Data Ingestion & Exploratory Data Analysis
### Topic Modelling on India News Headlines (2001–2023)

**Goal of this notebook:** load the raw dataset, understand its structure, and explore it
(volume over time, category distribution, headline length, vocabulary) *before* we touch any
modelling.

**Run this in Google Colab.**

> **Important — storage:** Colab gives each notebook session a fresh, temporary virtual machine.
> Anything saved to a plain relative path (like `../data/`) disappears once the session ends or
> a different notebook is opened. So this notebook mounts your **Google Drive** and saves all
> outputs there instead — that's what lets notebook 02 (in a separate session) pick up exactly
> where this one left off. The notebook files themselves still live in GitHub; only data/outputs
> live in Drive.


In [ ]:
# Mount Google Drive so files persist across separate Colab sessions/notebooks.
# The first time you run this, Colab will pop up a Google sign-in / permission prompt — approve it.
from google.colab import drive
drive.mount('/content/drive')

import os

# All data and output files for this project live under one folder in Drive.
# Defining the paths as variables (rather than hardcoding strings everywhere) means if the
# project folder ever moves, we only need to change it in one place.
BASE_DIR = '/content/drive/MyDrive/topic-modelling-capstone'
DATA_RAW = f'{BASE_DIR}/data/raw'
DATA_PROCESSED = f'{BASE_DIR}/data/processed'
OUTPUTS_FIGURES = f'{BASE_DIR}/outputs/figures'
OUTPUTS_MODELS = f'{BASE_DIR}/outputs/models'

# Create the folder structure if it doesn't exist yet (does nothing if it already does,
# thanks to exist_ok=True — safe to re-run this cell any number of times).
for d in [DATA_RAW, DATA_PROCESSED, OUTPUTS_FIGURES, OUTPUTS_MODELS]:
    os.makedirs(d, exist_ok=True)

print("Google Drive mounted. Working directory:", BASE_DIR)


In [ ]:
# Install/import the libraries this notebook needs.
# kagglehub: downloads the dataset directly from Kaggle (handles auth for us).
# pandas/numpy: data loading and manipulation.
# matplotlib/seaborn: plots for the EDA section.
# wordcloud: quick visual sanity-check of dominant vocabulary later on.
!pip install -q kagglehub pandas matplotlib seaborn wordcloud

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Fixing a random seed everywhere we sample data means anyone re-running this notebook gets
# the exact same sample — this is what "reproducible results" in the assignment actually means.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot styling — applied once, used for every chart in this notebook.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Get the dataset

Two options — set `USE_EXISTING_DRIVE_FILE` below accordingly:
- **`True`** — you already have the CSV in your Drive (e.g. previously downloaded manually).
  Just point `DRIVE_CSV_PATH` at it.
- **`False`** — download fresh via `kagglehub` (one-time browser login).

Using the file already in Drive is fine and reproducible as long as everyone on the team points
at the same file — just make sure `DRIVE_CSV_PATH` below matches where it actually sits in your
Drive (check via the Colab file browser sidebar, or Drive's "Copy path" option).


In [ ]:
# Toggle this depending on whether you already have the CSV in Drive, or need to fetch it.
USE_EXISTING_DRIVE_FILE = False   # False = download fresh via kagglehub

# Only used if USE_EXISTING_DRIVE_FILE is True — edit this to match your actual file location.
DRIVE_CSV_PATH = '/content/drive/MyDrive/india-news-headlines.csv'

if USE_EXISTING_DRIVE_FILE:
    # Just point csv_path at the file you already have — no download needed.
    csv_path = DRIVE_CSV_PATH
    assert os.path.exists(csv_path), (
        f"Couldn't find a file at {csv_path} — double check the path in the Colab file browser "
        f"(folder icon on the left sidebar) and update DRIVE_CSV_PATH above."
    )
    print("Using existing file in Drive:", csv_path)
else:
    # dataset_download() fetches the dataset (or reuses a cached copy) and returns the local
    # folder it was extracted to. This part uses Colab's local disk, not Drive, since we only
    # need the raw file once per session to build our working sample.
    path = kagglehub.dataset_download("therohk/india-headlines-news-dataset")
    print("Dataset downloaded to:", path)
    print(os.listdir(path))
    csv_path = os.path.join(path, "india-news-headlines.csv")


In [ ]:
# Load the CSV into a pandas DataFrame — this is the object we'll work with for the rest
# of the notebook.
df = pd.read_csv(csv_path)
print(f"Rows: {len(df):,} | Columns: {list(df.columns)}")
df.head()


## 2. First look — schema, missing values, duplicates

The dataset has three columns:
- `publish_date` — integer, format `YYYYMMDD`
- `headline_category` — dot-separated category taxonomy (e.g. `india`, `city.mumbai`, `sports.cricket`)
- `headline_text` — the headline itself


In [ ]:
# .info() shows column dtypes and non-null counts — quick check that nothing looks obviously broken.
df.info()

# isna().sum() counts missing values per column. If headline_text had many missing values,
# that would be a serious data quality issue worth investigating before modelling.
print("\nMissing values per column:")
print(df.isna().sum())

# duplicated() with no arguments flags rows that are identical across ALL columns.
# duplicated(subset='headline_text') flags rows with the same headline text even if published
# on a different date/category — this can happen with wire-service reprints or syndication.
print(f"\nExact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate headline_text (across dates): {df.duplicated(subset='headline_text').sum():,}")


In [ ]:
# publish_date arrives as an integer like 20140812 — parse it into an actual datetime so we
# can do time-based analysis (grouping by year/month, plotting trends, etc.).
df['publish_date'] = pd.to_datetime(df['publish_date'], format='%Y%m%d')
df['year'] = df['publish_date'].dt.year
df['month'] = df['publish_date'].dt.month

# describe() on a datetime column shows count/min/max/quartiles for the dates themselves
# (std is NaN here — pandas doesn't compute a standard deviation for raw dates in this view,
# which is expected and not an error).
df[['publish_date', 'year', 'month']].describe()


## 3. Volume of headlines over time

Understanding volume trends matters for modelling: if certain years dominate, a topic model run
on the full data will be biased toward those years' vocabulary. This also informs our sampling
strategy (stratify by year to keep the sample representative).


In [ ]:
# Count headlines per year and plot as a bar chart.
yearly_counts = df.groupby('year').size()

fig, ax = plt.subplots()
yearly_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Number of headlines per year')
ax.set_xlabel('Year')
ax.set_ylabel('Headline count')
plt.tight_layout()

# Save the figure to Drive (not the local Colab disk) so it survives after this session ends
# and can be pulled straight into the report.
plt.savefig(f'{OUTPUTS_FIGURES}/headlines_per_year.png', dpi=150)
plt.show()

print(yearly_counts.describe())


## 3b. Headline volume per month

A finer-grained view than the yearly plot in Section 3 — shows whether volume growth was smooth
or had sharper step-changes at specific points, and can reveal any monthly/seasonal patterns the
yearly aggregation would hide.


In [ ]:
df['year_month'] = df['publish_date'].dt.to_period('M')
monthly_counts = df.groupby('year_month').size()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_counts.plot(ax=ax, color='steelblue', linewidth=1)
ax.set_title('Number of headlines per month, 2001-2023')
ax.set_xlabel('Month')
ax.set_ylabel('Headline count')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/headlines_per_month.png', dpi=150)
plt.show()


## 4. Category distribution

`headline_category` is a dot-separated taxonomy (e.g. `sports.cricket`, `city.mumbai.civic`).
We'll look at the top-level category (before the first dot) as a coarse view.


In [ ]:
# Take everything before the first '.' as the coarse top-level category
# (e.g. "sports.cricket.ipl" -> "sports"). This collapses hundreds of fine-grained
# subcategories into a manageable number of high-level buckets for a first look.
df['category_top'] = df['headline_category'].astype(str).str.split('.').str[0]

# value_counts() ranks categories by frequency; head(20) keeps only the top 20 for readability.
top_categories = df['category_top'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_categories.plot(kind='barh', ax=ax, color='darkorange')
ax.invert_yaxis()  # so the largest category appears at the top of the chart, not the bottom
ax.set_title('Top 20 top-level categories by headline count')
ax.set_xlabel('Headline count')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/top_categories.png', dpi=150)
plt.show()

print(f"Number of distinct top-level categories: {df['category_top'].nunique()}")
print(f"Number of distinct full categories: {df['headline_category'].nunique()}")


## 5. Headline length distribution

Headline length (in words) affects preprocessing decisions (e.g. minimum token count to keep a
document) and gives a sanity check that the text looks like real headlines, not garbage rows.


In [ ]:
# Count words per headline by splitting on whitespace. astype(str) guards against any
# stray non-string/NaN values in the column causing an error here.
df['headline_word_count'] = df['headline_text'].astype(str).str.split().apply(len)

fig, ax = plt.subplots()
sns.histplot(df['headline_word_count'], bins=30, ax=ax, color='seagreen')
ax.set_title('Distribution of headline length (in words)')
ax.set_xlabel('Word count')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/headline_length_dist.png', dpi=150)
plt.show()

df['headline_word_count'].describe()


In [ ]:
# Eyeball the extremes — very short headlines might be junk/truncated rows; very long ones
# might indicate a parsing issue (e.g. two headlines concatenated together).
print("Shortest headlines:")
print(df.nsmallest(5, 'headline_word_count')[['headline_text', 'headline_word_count']])
print("\nLongest headlines:")
print(df.nlargest(5, 'headline_word_count')[['headline_text', 'headline_word_count']])


## 5b. Headline length in characters

In addition to word count, character length is a useful complementary view — especially since
headline character limits (common in print/wire journalism) can create visible ceiling effects
that word count alone doesn't show as clearly.


In [ ]:
df['headline_char_count'] = df['headline_text'].astype(str).str.len()

fig, ax = plt.subplots()
sns.histplot(df['headline_char_count'], bins=40, ax=ax, color='coral')
ax.set_title('Distribution of headline length (in characters)')
ax.set_xlabel('Character count')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/headline_length_chars_dist.png', dpi=150)
plt.show()

df['headline_char_count'].describe()


## 5c. Vocabulary size

The number of *unique* words in the corpus — a basic but important measure of lexical diversity,
and relevant context for topic modelling (a larger vocabulary means more distinct terms c-TF-IDF
has to work with when building topic keyword lists later).


In [ ]:
import re
from collections import Counter

# Simple whitespace + lowercase tokenization for a quick vocabulary count — deliberately not
# using the full heavy-cleaning pipeline here (that happens properly in notebook 02); this is
# just a first-look EDA statistic on the raw text.
def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text.split()

# Uses a fresh random 300k-headline sample (not the stratified-by-year sample built in Section 7 —
# this section runs earlier in the notebook, and vocabulary size / word frequencies don't need
# year-stratification the way modelling does; a plain random sample is sufficient and stabilizes
# well before using all 3.8M rows).
vocab_sample = df['headline_text'].astype(str).sample(min(300_000, len(df)), random_state=RANDOM_STATE)
all_tokens = [tok for text in vocab_sample for tok in simple_tokenize(text)]

vocab = set(all_tokens)
print(f"Total tokens (300k-headline sample): {len(all_tokens):,}")
print(f"Vocabulary size (unique words): {len(vocab):,}")


## 5d. Most frequent unigrams and bigrams

Ranked frequency tables of single words (unigrams) and two-word phrases (bigrams) — a more
precise complement to the word cloud in Section 6, useful for spotting both common individual
terms and common short phrases (e.g. "world cup", "prime minister") that a unigram-only view
would miss.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Self-contained: recreate the same 300k random sample here (same seed = identical sample)
# rather than relying on a variable from a previous cell — makes this cell runnable on its
# own regardless of whether earlier cells were run first, or whether the runtime restarted.
vocab_sample = df['headline_text'].astype(str).sample(min(300_000, len(df)), random_state=RANDOM_STATE)
sample_text_list = vocab_sample.tolist()

# Unigrams, English stopwords removed so the ranking reflects topical words rather than
# function words (the, and, of, ...).
uni_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 1), max_features=20)
uni_counts = uni_vectorizer.fit_transform(sample_text_list).sum(axis=0)
uni_freq = sorted(zip(uni_vectorizer.get_feature_names_out(), uni_counts.tolist()[0]), key=lambda x: -x[1])

print("Top 20 unigrams:")
for word, count in uni_freq:
    print(f"  {word:<20} {count:,}")

# Bigrams
bi_vectorizer = CountVectorizer(stop_words='english', ngram_range=(2, 2), max_features=20)
bi_counts = bi_vectorizer.fit_transform(sample_text_list).sum(axis=0)
bi_freq = sorted(zip(bi_vectorizer.get_feature_names_out(), bi_counts.tolist()[0]), key=lambda x: -x[1])

print("\nTop 20 bigrams:")
for phrase, count in bi_freq:
    print(f"  {phrase:<25} {count:,}")


## 6. Quick vocabulary / word-frequency glance

A rough word cloud gives an intuitive first read on dominant themes before any formal modelling —
useful as a sanity check against the topics BERTopic eventually surfaces.


In [ ]:
from wordcloud import WordCloud, STOPWORDS

# Sampling 200k headlines (rather than joining all 3.8M) keeps this fast while still giving
# a representative picture of common vocabulary. random_state fixes which rows get picked,
# so this is reproducible.
sample_text = " ".join(df['headline_text'].astype(str).sample(200_000, random_state=RANDOM_STATE))

# STOPWORDS here removes common English filler words (the, and, of, ...) so the word cloud
# highlights actual topical content instead of function words.
wc = WordCloud(width=1200, height=600, background_color='white',
                stopwords=STOPWORDS, max_words=150).generate(sample_text)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word cloud — 200k sampled headlines (raw, unprocessed)')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/wordcloud_raw_sample.png', dpi=150)
plt.show()


## 7. Build the stratified working sample

The full dataset (~3.8M rows) is too large to iterate on quickly. We build a
**stratified-by-year sample** so every year 2001–2023 is proportionally represented, keeping the
topic distribution over time realistic. This sample is what we'll use for the main preprocessing
+ modelling + hyperparameter experiments in the next notebooks.

> **Reproducibility:** the sample size and `RANDOM_STATE` are fixed here — rerunning this cell
> always produces the same sample.


In [ ]:
# Target sample size — ~8% of the full dataset. Large enough to give BERTopic a solid,
# representative corpus to find topics in, small enough to run comfortably on Colab.
# Adjust this up/down based on how much time your runs are taking.
SAMPLE_SIZE = 300_000

# groupby('year').apply(sample) draws the SAME proportion of rows from every year, so a year
# with more headlines contributes proportionally more to the sample — this is what "stratified"
# means here, and it's what keeps the sample's year distribution matching the full dataset's.
sample_df = (
    df.groupby('year', group_keys=False)
      .apply(lambda x: x.sample(frac=SAMPLE_SIZE / len(df), random_state=RANDOM_STATE))
      .reset_index(drop=True)
)
# Note: this triggers a harmless pandas DeprecationWarning about grouping columns in newer
# versions — safe to ignore; the sampling logic and output are correct either way.

print(f"Full dataset: {len(df):,} rows")
print(f"Sampled dataset: {len(sample_df):,} rows")
print("\nSample year distribution (should mirror full dataset proportions):")
print((sample_df['year'].value_counts(normalize=True).sort_index() * 100).round(2))


In [ ]:
# Save the sample to Drive (persistent) so notebook 02 — running in a completely separate
# Colab session — can load it directly by reading from the same DATA_PROCESSED path.
sample_df.to_csv(f'{DATA_PROCESSED}/headlines_sample_300k.csv', index=False)
print(f"Saved to {DATA_PROCESSED}/headlines_sample_300k.csv")
print("This file now persists in your Google Drive — notebook 02 will load it from there.")


## 8. Summary of EDA findings

- **Total headlines: 3,876,557**, spanning **2001–2023** (23 years, 2023 partial — data ends June 2023).
- **Yearly volume trend**: grows steadily from ~57k headlines (2001) to a plateau of ~250-255k/year
  during 2012-2018, then dips to ~180k/year (2019-2022), with 2023 lower still due to being a
  partial year.
- **Monthly volume trend reveals a sharper pattern than the yearly view**: the drop from the
  2012-2018 plateau (~21,500 headlines/month) to the 2019-2022 level (~15,000/month) happens as a
  **sharp step-change right at the 2019 boundary**, not a gradual decline — suggesting a specific
  publishing or archival policy change around that time rather than a slow real-world drop in
  news volume. A small, consistent sawtooth oscillation is also visible across the plateau years,
  plausibly a seasonal/editorial-calendar effect.
- **Missing values: zero** across all three columns — a very clean dataset by that measure.
- **Duplicates**: 25,645 exact duplicate rows (~0.66% of the dataset), and 271,802 headlines
  (~7%) repeated across different dates/categories — likely wire reprints or category
  re-tagging of the same story. Addressed by deduplication in notebook 02.
- **Dominant categories**: 336 distinct top-level categories, 1,024 full categories — a long
  tail. `city` dominates overwhelmingly (~2.3M headlines, ~60% of the dataset), followed by
  `india`, `entertainment`, `unknown`, `business`, `sports`.
- **Typical headline length (words)**: mean 7.76, median 8, std 2.7, range 1-32 words.
- **Typical headline length (characters)**: mean 46.9, median 45, std 16.3, range 6-130
  characters — consistent with a print/wire-style headline length convention.
- **Vocabulary size**: 73,207 unique words found in a 300k-headline random sample (2,347,219
  total tokens) — a substantial lexical diversity for c-TF-IDF topic keyword extraction to work
  with later.
- **Most frequent unigrams**: india, rs, new, govt, held, man, says, delhi, bjp, year, city,
  case, police, day, hc, cops, old, mumbai, court, gets — dominated by generic
  news-reporting/location terms, as expected for a raw, unprocessed sample (motivates the
  stopword removal step in notebook 02).
- **Most frequent bigrams**: "year old", "tamil nadu", "covid 19", "high court", "rs lakh",
  "world cup", "andhra pradesh", "municipal corporation", "narendra modi" — bigrams surface more
  topically specific phrases than unigrams alone (locations, named entities, recurring news
  formats), an early hint at some of the themes the topic model later discovers independently
  (e.g. COVID, cricket/World Cup, politics).
- **Data quality note**: a small number of junk/non-headline rows exist (e.g. one row is
  keyboard-mash text, "a s d f g h j k l m n..."). Minor, but confirms the dataset isn't
  perfectly clean — the token-count filter in notebook 02 helps guard against this kind of noise.
- **Sample validity**: the stratified 300k sample's year proportions closely track the full
  dataset's (e.g. 2015 is ~6.6% of both) — confirms the sampling strategy preserved the temporal
  distribution as intended.
